In [ ]:
# ==============================================================================
# NETWORK ROBUSTNESS
# ==============================================================================
import os
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "10")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")
os.environ["HF_HUB_DISABLE_XET"] = "1"
# os.environ["HF_TOKEN"] = "hf_..."   # uncomment + fill in if you hit rate limits

# !pip install torch transformers datasets accelerate
import re
import sys
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
from typing import Dict, Tuple, List, Set
import gc
import time

# ==============================================================================
# CONFIGURATION
# ==============================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LEN = 2048
PAGE_SIZE = 64
W_PAGES = 2
TOP_K_PAGES = 4
EVAL_QUERIES = 256

LAYERS_TO_HOOK = [4, 10, 16]
FRACTION_SWEEP = [0.05, 0.10, 0.25, 0.50, 0.75, 1.0]

# Default (primary-report) labeling config.
DEFAULT_CONTEXT_TOKENS = 64
DEFAULT_PERCENTILE = 50

# >>> NEW: sensitivity sweep grid for the Q4-labeling parameters introduced by
# the circularity fix. These were never part of the original proposal, so the
# headline result needs to survive being re-labeled under different choices.
SENSITIVITY_CONTEXT_TOKENS = [32, 64, 128]
SENSITIVITY_PERCENTILES = [25, 50, 75]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}", flush=True)

# ==============================================================================
# 1. LOAD MODEL & DATA
# ==============================================================================
def load_with_retry(load_fn, name, max_retries=3, backoff=5):
    for attempt in range(1, max_retries + 1):
        try:
            return load_fn()
        except Exception as e:
            print(f"[{name}] attempt {attempt}/{max_retries} failed: {type(e).__name__}: {e}", flush=True)
            if attempt == max_retries:
                raise
            print(f"Retrying in {backoff}s...", flush=True)
            time.sleep(backoff)

print(f"Loading {MODEL_NAME}...", flush=True)
tokenizer = load_with_retry(lambda: AutoTokenizer.from_pretrained(MODEL_NAME), "tokenizer")
model = load_with_retry(
    lambda: AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto"),
    "model"
)
model.eval()

print("Loading real long-context dataset (wikitext)...", flush=True)
dataset = load_with_retry(
    lambda: load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test"),
    "dataset"
)
text = "\n".join(dataset["text"])
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(device)
seq_len = inputs.input_ids.shape[1]

# ==============================================================================
# 2. EXTRACT RUNTIME TENSORS (per-layer)
# ==============================================================================
def make_hook(store: dict):
    def llama_attention_hook(module, args, kwargs, output):
        hidden_states = args[0] if len(args) > 0 else kwargs['hidden_states']
        bsz, q_len, _ = hidden_states.size()

        cfg = module.config
        num_heads = getattr(module, 'num_heads', None) or cfg.num_attention_heads
        num_kv_heads = getattr(module, 'num_key_value_heads', None) or getattr(cfg, 'num_key_value_heads', num_heads)
        head_dim = getattr(module, 'head_dim', None) or (cfg.hidden_size // cfg.num_attention_heads)
        n_rep = num_heads // num_kv_heads

        query_states = module.q_proj(hidden_states)
        key_states = module.k_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)
        if n_rep > 1:
            key_states = key_states.repeat_interleave(n_rep, dim=1)

        store['pre_rope_k'] = key_states.detach().clone()

        if kwargs.get('position_embeddings', None) is not None:
            cos, sin = kwargs['position_embeddings']
            cos, sin = cos[:, :q_len], sin[:, :q_len]
        else:
            position_ids = kwargs.get('position_ids', None)
            if position_ids is None:
                position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
            cos, sin = module.rotary_emb(key_states, position_ids[:, :q_len])

        def apply_rotary(x, cos, sin):
            cos_b = cos.unsqueeze(1)
            sin_b = sin.unsqueeze(1)
            x1 = x[..., : x.shape[-1] // 2]
            x2 = x[..., x.shape[-1] // 2 :]
            rotated = torch.cat((-x2, x1), dim=-1)
            return (x * cos_b) + (rotated * sin_b)

        query_states = apply_rotary(query_states, cos, sin)
        post_rope_k = apply_rotary(key_states, cos, sin)

        store['q'] = query_states.detach().clone()
        store['post_rope_k'] = post_rope_k.detach().clone()

        attn_weights = torch.matmul(query_states, post_rope_k.transpose(2, 3)) / np.sqrt(head_dim)
        mask = torch.triu(torch.ones(q_len, q_len, dtype=torch.bool, device=device), diagonal=1)
        attn_weights.masked_fill_(mask, float('-inf'))
        attn_probs = F.softmax(attn_weights, dim=-1)
        store['exact_attn'] = attn_probs.detach().clone()
    return llama_attention_hook

per_layer_tensors = {}
handles = []
for layer_idx in LAYERS_TO_HOOK:
    store = {}
    per_layer_tensors[layer_idx] = store
    h = model.model.layers[layer_idx].self_attn.register_forward_hook(make_hook(store), with_kwargs=True)
    handles.append(h)

print(f"Running forward pass on {seq_len} tokens to capture exact state...", flush=True)
with torch.no_grad():
    model(**inputs)
for h in handles:
    h.remove()

input_ids_cpu = inputs.input_ids[0].detach().cpu()

del model
gc.collect()
torch.cuda.empty_cache()

# ==============================================================================
# 2b. INDEPENDENT CONTENT-SIMILARITY SIGNAL (lexical, model-free)
# ==============================================================================
# Model-free word-overlap signal. Fully decoupled from Q/K vectors and from
# every routing method's own score, so it cannot structurally favor/penalize
# any of A/B/C when used to define the Q1-Q4 quadrants.
_word_re = re.compile(r"\w+")

def _word_set(token_ids: torch.Tensor) -> Set[str]:
    txt = tokenizer.decode(token_ids.tolist(), skip_special_tokens=True)
    return set(w.lower() for w in _word_re.findall(txt))

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 0.0
    union = len(a | b)
    return len(a & b) / union if union else 0.0

num_pages_total = seq_len // PAGE_SIZE
print(f"Precomputing lexical word-sets for {num_pages_total} pages...", flush=True)
page_word_sets: List[Set[str]] = []
for p in range(num_pages_total):
    start, end = p * PAGE_SIZE, (p + 1) * PAGE_SIZE
    page_word_sets.append(_word_set(input_ids_cpu[start:end]))

query_indices_global = list(range(seq_len - EVAL_QUERIES, seq_len))

# >>> NEW: everything parameterized by context_tokens so the sensitivity sweep
# can cheaply relabel without touching the model or the routing scores at all.
query_ctx_caches: Dict[int, Dict[int, Set[str]]] = {}
lexical_sim_caches: Dict[int, Dict[Tuple[int, int], float]] = {}
all_sims_cache: Dict[int, List[float]] = {}

def build_context_signal(context_tokens: int):
    if context_tokens in query_ctx_caches:
        return
    print(f"  Building lexical context signal for context_tokens={context_tokens}...", flush=True)
    ctx_cache: Dict[int, Set[str]] = {}
    for q_idx in query_indices_global:
        start = max(0, q_idx - context_tokens + 1)
        ctx_cache[q_idx] = _word_set(input_ids_cpu[start:q_idx + 1])
    query_ctx_caches[context_tokens] = ctx_cache

    sim_cache: Dict[Tuple[int, int], float] = {}
    all_sims: List[float] = []
    for q_idx in query_indices_global:
        current_page_idx = q_idx // PAGE_SIZE
        tier1_start_page = max(0, current_page_idx - W_PAGES)
        for p in range(tier1_start_page):
            v = jaccard(ctx_cache[q_idx], page_word_sets[p])
            sim_cache[(q_idx, p)] = v
            all_sims.append(v)
    lexical_sim_caches[context_tokens] = sim_cache
    all_sims_cache[context_tokens] = all_sims

def get_lexical_sim(context_tokens: int, q_idx: int, page_idx: int) -> float:
    return lexical_sim_caches[context_tokens][(q_idx, page_idx)]

def get_threshold(context_tokens: int, percentile: float) -> float:
    return float(np.percentile(all_sims_cache[context_tokens], percentile))

for ct in sorted(set(SENSITIVITY_CONTEXT_TOKENS + [DEFAULT_CONTEXT_TOKENS])):
    build_context_signal(ct)

DEFAULT_THRESHOLD = get_threshold(DEFAULT_CONTEXT_TOKENS, DEFAULT_PERCENTILE)
print(f"Default lexical-similarity threshold (context={DEFAULT_CONTEXT_TOKENS}, "
      f"p{DEFAULT_PERCENTILE}): {DEFAULT_THRESHOLD:.4f}", flush=True)

# ==============================================================================
# 3. ROUTING ENGINE
# ==============================================================================
def get_low_freq_indices(dim: int, fraction: float) -> torch.Tensor:
    num_dims_to_keep = max(1, int((dim // 2) * fraction))
    first_half_indices = torch.arange((dim // 2) - num_dims_to_keep, dim // 2)
    second_half_indices = torch.arange(dim - num_dims_to_keep, dim)
    return torch.cat([first_half_indices, second_half_indices]).to(device)

def compute_page_bounds(k_tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    valid_len = (k_tensor.shape[1] // PAGE_SIZE) * PAGE_SIZE
    k_trunc = k_tensor[:, :valid_len, :]
    k_pages = k_trunc.view(k_trunc.shape[0], -1, PAGE_SIZE, k_trunc.shape[2])
    return k_pages.min(dim=2)[0], k_pages.max(dim=2)[0]

def score_pages_quest_style(q_slice, page_mins, page_maxs) -> torch.Tensor:
    q_pos = F.relu(q_slice).unsqueeze(1)
    q_neg = -F.relu(-q_slice).unsqueeze(1)
    return (q_pos * page_maxs).sum(dim=-1) + (q_neg * page_mins).sum(dim=-1)

# ==============================================================================
# 4. PHASE 1 — ROUTING DECISIONS (independent of Q4 labeling; computed ONCE
#    per layer/method, then reused across the whole sensitivity grid)
# ==============================================================================
def compute_routing_decisions(q, k_pre, k_post, exact_attn):
    """Returns, per method, a list of per-query dicts holding everything needed
    to score Q1-Q4/recall EXCEPT the similarity labeling itself. This is the
    key refactor vs. the previous script: scoring/selection and quadrant
    labeling are now decoupled, so relabeling for the sensitivity sweep is
    cheap (no rerouting, no model calls)."""
    num_heads, seq_len_local, head_dim = q.shape
    num_pages = seq_len_local // PAGE_SIZE
    page_centers = torch.arange(num_pages, device=device) * PAGE_SIZE + (PAGE_SIZE // 2)
    valid_len = (k_pre.shape[1] // PAGE_SIZE) * PAGE_SIZE
    k_pre_pages = k_pre[:, :valid_len, :].view(num_heads, num_pages, PAGE_SIZE, head_dim)
    page_means_pre = k_pre_pages.mean(dim=2)

    query_indices = list(range(seq_len_local - EVAL_QUERIES, seq_len_local))
    bounds_A = compute_page_bounds(k_post)

    decisions = {}
    for method in ['A', 'C'] + [f'B_{frac}' for frac in FRACTION_SWEEP]:
        per_query = []
        bound_widths = []
        for q_idx in query_indices:
            q_head_vecs = q[:, q_idx, :]
            current_page_idx = q_idx // PAGE_SIZE
            tier1_start_page = max(0, current_page_idx - W_PAGES)
            tier1_pages = list(range(tier1_start_page, current_page_idx + 1))
            routable_pages_idx = torch.arange(0, tier1_start_page, device=device)
            if len(routable_pages_idx) <= TOP_K_PAGES:
                continue

            if method == 'A':
                scores = score_pages_quest_style(
                    q_head_vecs, bounds_A[0][:, routable_pages_idx, :], bounds_A[1][:, routable_pages_idx, :])
            elif method == 'C':
                q_norm = F.normalize(q_head_vecs.unsqueeze(1), dim=-1)
                p_norm = F.normalize(page_means_pre[:, routable_pages_idx, :], dim=-1)
                scores = (q_norm * p_norm).sum(dim=-1)
            else:
                fraction = float(method.split('_')[1])
                idx = get_low_freq_indices(head_dim, fraction)
                q_sub = q_head_vecs[:, idx]
                mins_sub = bounds_A[0][:, routable_pages_idx][:, :, idx]
                maxs_sub = bounds_A[1][:, routable_pages_idx][:, :, idx]
                scores = score_pages_quest_style(q_sub, mins_sub, maxs_sub)
                bound_widths.append((maxs_sub - mins_sub).mean().item())

            topk_indices = scores.topk(TOP_K_PAGES, dim=-1).indices
            selected_pages = routable_pages_idx[topk_indices]  # [num_heads, TOP_K]

            dists_all = (q_idx - page_centers[routable_pages_idx].float()).abs()
            dist_median = dists_all.median().item()
            is_far_per_page = dists_all > dist_median

            gt_attn = exact_attn[:, q_idx, :]  # [num_heads, seq_len]
            gt_page_mass = torch.zeros(num_heads, len(routable_pages_idx), device=device)
            for i, p in enumerate(routable_pages_idx.tolist()):
                start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
                gt_page_mass[:, i] = gt_attn[:, start_tok:end_tok].sum(dim=-1)

            tier1_mass = 0.0
            for p in tier1_pages:
                start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
                tier1_mass += gt_attn[:, start_tok:end_tok].sum().item()

            selected_mass = 0.0
            for h in range(num_heads):
                for p in selected_pages[h].tolist():
                    start_tok, end_tok = p * PAGE_SIZE, min((p + 1) * PAGE_SIZE, q_idx + 1)
                    selected_mass += gt_attn[h, start_tok:end_tok].sum().item()

            per_query.append({
                'q_idx': q_idx,
                'routable_pages_idx': routable_pages_idx,
                'selected_pages': selected_pages,
                'gt_page_mass': gt_page_mass,
                'is_far_per_page': is_far_per_page,
                'recall_mass_sum': selected_mass + tier1_mass,
                'recall_evals': num_heads,
            })
        decisions[method] = {'per_query': per_query, 'bound_widths': bound_widths}
    return decisions

# ==============================================================================
# 5. PHASE 2 — Q4 LABELING & AGGREGATION (cheap; reruns per sensitivity combo)
# ==============================================================================
def score_with_labeling(decisions, context_tokens: int, percentile: float):
    threshold = get_threshold(context_tokens, percentile)
    results = {}
    for method, d in decisions.items():
        total_recall, num_recall_evals = 0.0, 0
        q4_samples, q4_eligible = [], 0
        q_stats = {'Q1': [], 'Q2': [], 'Q3': []}

        for rec in d['per_query']:
            q_idx = rec['q_idx']
            routable = rec['routable_pages_idx'].tolist()
            is_far = rec['is_far_per_page']
            gt_page_mass = rec['gt_page_mass']
            selected_pages = rec['selected_pages']
            num_heads = gt_page_mass.shape[0]

            total_recall += rec['recall_mass_sum']
            num_recall_evals += rec['recall_evals']

            lex_sims = torch.tensor(
                [get_lexical_sim(context_tokens, q_idx, p) for p in routable], device=device)
            is_high_sim = lex_sims > threshold

            for h in range(num_heads):
                page_mass_h = gt_page_mass[h]
                mass_median = page_mass_h.median()
                is_important = page_mass_h > mass_median
                true_q4_mask = is_important & is_far & (~is_high_sim)
                true_q4_target_mass = page_mass_h[true_q4_mask].sum().item()

                selected_p = selected_pages[h].tolist()
                if true_q4_target_mass > 0:
                    q4_eligible += 1
                    recovered = sum(
                        page_mass_h[i].item()
                        for i, p in enumerate(routable)
                        if p in selected_p and true_q4_mask[i]
                    )
                    q4_samples.append(recovered / true_q4_target_mass)

                for p in selected_p:
                    if p not in routable:
                        continue
                    i = routable.index(p)
                    far = is_far[i].item()
                    high_sim = is_high_sim[i].item()
                    pm = page_mass_h[i].item()
                    if not far and high_sim: q_stats['Q1'].append(pm)
                    elif not far and not high_sim: q_stats['Q2'].append(pm)
                    elif far and high_sim: q_stats['Q3'].append(pm)

        results[method] = {
            'recall': (total_recall / num_recall_evals) * 100 if num_recall_evals else 0.0,
            'Q4_target_recovery': np.mean(q4_samples) * 100 if q4_samples else 0.0,
            'Q4_eligible_count': q4_eligible,
            'Q1_mass': np.mean(q_stats['Q1']) if q_stats['Q1'] else 0.0,
            'Q2_mass': np.mean(q_stats['Q2']) if q_stats['Q2'] else 0.0,
            'Q3_mass': np.mean(q_stats['Q3']) if q_stats['Q3'] else 0.0,
            'avg_bound_width': np.mean(d['bound_widths']) if d['bound_widths'] else None,
        }
    return results, threshold

# ==============================================================================
# 6. EXECUTION — PRIMARY REPORT (default labeling) PER LAYER
# ==============================================================================
all_layer_decisions = {}
all_layer_primary_results = {}

for layer_idx in LAYERS_TO_HOOK:
    t0 = time.time()
    print("\n" + "=" * 70, flush=True)
    print(f"STARTING ROUTING EXPERIMENT — LAYER {layer_idx}", flush=True)
    print("=" * 70, flush=True)

    store = per_layer_tensors[layer_idx]
    q, k_pre, k_post, exact_attn = store['q'][0], store['pre_rope_k'][0], store['post_rope_k'][0], store['exact_attn'][0]

    decisions = compute_routing_decisions(q, k_pre, k_post, exact_attn)
    all_layer_decisions[layer_idx] = decisions

    results, thr = score_with_labeling(decisions, DEFAULT_CONTEXT_TOKENS, DEFAULT_PERCENTILE)
    all_layer_primary_results[layer_idx] = results

    print(f"\nTier-1 Local Window: {W_PAGES} pages ({W_PAGES*PAGE_SIZE} tokens) - Always Loaded")
    print(f"Tier-2 Routing: Selecting {TOP_K_PAGES} pages from history")
    print(f"Q1-Q4 split: LEXICAL similarity, context={DEFAULT_CONTEXT_TOKENS} tok, "
          f"p{DEFAULT_PERCENTILE} threshold={thr:.4f}")
    print("-" * 100)
    print(f"{'Method':<12} | {'Subset %':<9} | {'Recall':<8} | {'Q1':<7} | {'Q2':<7} | {'Q3':<7} | {'Q4 TargetRecov':<15} | {'Q4 elig':<8} | {'BoundW'}")
    print("-" * 100)

    r = results['C']
    print(f"Baseline C | {'Pre-RoPE':<9} | {r['recall']:5.2f}%  | {r['Q1_mass']:.3f}  | {r['Q2_mass']:.3f}  | {r['Q3_mass']:.3f}  | "
          f"{r['Q4_target_recovery']:6.2f}%        | {r['Q4_eligible_count']:<8}| N/A")
    for frac in FRACTION_SWEEP:
        r = results[f'B_{frac}']
        bw = f"{r['avg_bound_width']:.3f}" if r['avg_bound_width'] is not None else "N/A"
        print(f"Method B     | {frac*100:6.1f}%  | {r['recall']:5.2f}%  | {r['Q1_mass']:.3f}  | {r['Q2_mass']:.3f}  | {r['Q3_mass']:.3f}  | "
              f"{r['Q4_target_recovery']:6.2f}%        | {r['Q4_eligible_count']:<8}| {bw}")
    r = results['A']
    print(f"Baseline A | {'100 (Full)':<9} | {r['recall']:5.2f}%  | {r['Q1_mass']:.3f}  | {r['Q2_mass']:.3f}  | {r['Q3_mass']:.3f}  | "
          f"{r['Q4_target_recovery']:6.2f}%        | {r['Q4_eligible_count']:<8}| N/A")

    best_B_frac = sorted(FRACTION_SWEEP, key=lambda f: results[f'B_{f}']['Q4_target_recovery'], reverse=True)[0]
    q4_C, q4_B_best, c_elig = results['C']['Q4_target_recovery'], results[f'B_{best_B_frac}']['Q4_target_recovery'], results['C']['Q4_eligible_count']
    if c_elig == 0:
        print(f"\n[INCONCLUSIVE] 0 eligible Q4 comparisons at layer {layer_idx}.")
    elif q4_B_best > q4_C:
        print(f"\n[SUCCESS] B (at {best_B_frac*100}%) recovers {q4_B_best:.2f}% vs C's {q4_C:.2f}% (n={c_elig}).")
    else:
        print(f"\n[NEGATIVE RESULT] B's best ({q4_B_best:.2f}%) did not exceed C's {q4_C:.2f}% (n={c_elig}).")

    print(f"\n[layer {layer_idx} done in {time.time()-t0:.1f}s]", flush=True)

# ==============================================================================
# 7. CROSS-LAYER SUMMARY (default labeling)
# ==============================================================================
print("\n" + "=" * 70)
print("CROSS-LAYER SUMMARY — Q4 Target Recovery (%) by method (default labeling)")
print("=" * 70)
header = f"{'Layer':<8}" + "".join(f"{f'B_{int(f*100)}%':<10}" for f in FRACTION_SWEEP) + f"{'A(100%)':<10}{'C(PreRoPE)':<12}"
print(header)
for layer_idx in LAYERS_TO_HOOK:
    r = all_layer_primary_results[layer_idx]
    row = f"{layer_idx:<8}"
    for f in FRACTION_SWEEP:
        row += f"{r[f'B_{f}']['Q4_target_recovery']:<10.2f}"
    row += f"{r['A']['Q4_target_recovery']:<10.2f}{r['C']['Q4_target_recovery']:<12.2f}"
    print(row)

# ==============================================================================
# 8. >>> NEW: SENSITIVITY SWEEP — does B > C in Q4 survive different labeling
#    choices? Reuses cached routing decisions (Section 6/Phase 1); only the
#    quadrant labeling is recomputed, so this is fast (no rerouting).
# ==============================================================================
print("\n" + "=" * 70)
print("SENSITIVITY SWEEP — B-vs-C Q4 gap under different labeling parameters")
print("(context_tokens x threshold_percentile; B compared at ITS OWN best")
print(" fraction for each combo, since the optimal fraction can shift)")
print("=" * 70)

sensitivity_summary = []  # (layer, context_tokens, percentile, q4_C, q4_B_best, best_frac, c_elig, verdict)

for layer_idx in LAYERS_TO_HOOK:
    decisions = all_layer_decisions[layer_idx]
    print(f"\n--- Layer {layer_idx} ---")
    print(f"{'ctx_tok':<8} | {'pctile':<7} | {'thresh':<8} | {'C Q4%':<8} | {'B best Q4%':<11} | {'best frac':<10} | {'C elig':<8} | verdict")
    for ct in SENSITIVITY_CONTEXT_TOKENS:
        for pct in SENSITIVITY_PERCENTILES:
            results, thr = score_with_labeling(decisions, ct, pct)
            best_frac = sorted(FRACTION_SWEEP, key=lambda f: results[f'B_{f}']['Q4_target_recovery'], reverse=True)[0]
            q4_C = results['C']['Q4_target_recovery']
            q4_B = results[f'B_{best_frac}']['Q4_target_recovery']
            c_elig = results['C']['Q4_eligible_count']
            if c_elig == 0:
                verdict = "INCONCLUSIVE"
            elif q4_B > q4_C:
                verdict = "B WINS"
            else:
                verdict = "C WINS/TIE"
            sensitivity_summary.append((layer_idx, ct, pct, q4_C, q4_B, best_frac, c_elig, verdict))
            print(f"{ct:<8} | {pct:<7} | {thr:<8.4f} | {q4_C:<8.2f} | {q4_B:<11.2f} | {best_frac*100:<10.1f}| {c_elig:<8} | {verdict}")

n_total = len(sensitivity_summary)
n_b_wins = sum(1 for s in sensitivity_summary if s[-1] == "B WINS")
n_inconclusive = sum(1 for s in sensitivity_summary if s[-1] == "INCONCLUSIVE")
n_c_wins = n_total - n_b_wins - n_inconclusive

print("\n" + "=" * 70)
print("SENSITIVITY VERDICT SUMMARY")
print("=" * 70)
print(f"Total (layer x context x percentile) combos tested: {n_total}")
print(f"  B WINS:        {n_b_wins} ({100*n_b_wins/n_total:.0f}%)")
print(f"  C WINS/TIE:    {n_c_wins} ({100*n_c_wins/n_total:.0f}%)")
print(f"  INCONCLUSIVE:  {n_inconclusive} ({100*n_inconclusive/n_total:.0f}%)")
if n_b_wins == n_total - n_inconclusive and n_b_wins > 0:
    print("\n=> Result is ROBUST to the labeling-parameter choices tested: B beats C")
    print("   in Q4 across every non-inconclusive combo.")
elif n_b_wins > n_c_wins:
    print("\n=> Result LEANS toward B but is NOT uniform — some (context, percentile)")
    print("   combos flip the outcome. Report this as a boundary condition, not as")
    print("   an unconditional win.")
else:
    print("\n=> Result does NOT hold up under relabeling — the primary-report B-vs-C")
    print("   gap may be an artifact of the specific (context_tokens, percentile)")
    print("   choice used for the headline numbers. Treat the earlier [SUCCESS]")
    print("   verdicts as provisional pending this finding.")

Running on device: cuda
Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading real long-context dataset (wikitext)...
Running forward pass on 2048 tokens to capture exact state...
Precomputing lexical word-sets for 32 pages...
  Building lexical context signal for context_tokens=32...
  Building lexical context signal for context_tokens=64...
  Building lexical context signal for context_tokens=128...
Default lexical-similarity threshold (context=64, p50): 0.0526

STARTING ROUTING EXPERIMENT — LAYER 4

Tier-1 Local Window: 2 pages (128 tokens) - Always Loaded
Tier-2 Routing: Selecting 4 pages from history
Q1-Q4 split: LEXICAL similarity, context=64 tok, p50 threshold=0.0526
----------------------------------------------------------------------------------------------------
Method       | Subset %  | Recall   | Q1      | Q2      | Q3      | Q4 TargetRecov  | Q4 elig  | BoundW
----------------------------------------------------------------------------------------------------
Baseline C | Pre-RoPE  | 13.12%  | 0.003  | 0.003  | 0.017  |   7.77%        | 66